# 13. Recomendaciones Top-K Reales por Producto-Semilla

Reemplaza los ejemplos **ilustrativos** de la Tabla 18 por **salidas reales** del modelo. Para una lista curada de productos-semilla se fija un **contexto representativo** (ciudad y ruta modales, geolocalización mediana, cliente de alta frecuencia) y se recupera el top-K del catálogo por producto punto sobre los embeddings entrenados. El contexto fijo se documenta explícitamente: la complementariedad proviene del producto-semilla, no del cliente.

In [1]:
import os
# Kernel CWD robusto: subir hasta la raíz del proyecto (donde vive data_processed/)
_d = os.getcwd()
while not os.path.isdir("data_processed") and os.path.dirname(_d) != _d:
    os.chdir(".."); _d = os.getcwd()
assert os.path.isdir("data_processed"), f"No se encontró data_processed/ desde {os.getcwd()}"
print("CWD del kernel:", os.getcwd())

CWD del kernel: /Users/allan/Documents/maestria/tesis v4


In [2]:
import polars as pl
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, os
np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")
K_LIST = [10, 50, 100]
print("TF", tf.__version__, "| Polars", pl.__version__)

TF 2.21.0 | Polars 1.41.2


In [3]:
train_df = pl.read_parquet("data_processed/retrieval_train.parquet")
products_df = pl.read_parquet("data_processed/products_catalog.parquet").unique(subset=["id_producto"])
products_df = products_df.with_columns([
    pl.col("marca").fill_null("SIN_MARCA"), pl.col("familia1").fill_null("SIN_CATEGORIA"),
    pl.col("familia2").fill_null("SIN_SUBCATEGORIA"), pl.col("precio").fill_null(0.0),
    pl.col("peso_unitario").fill_null(0.0),
    pl.col("descripción corta").fill_null("(sin descripción)").alias("desc")])
catalog_ids = products_df["id_producto"].to_list()
catalog_arr = np.array(catalog_ids)
desc_map = {r["id_producto"]: r["desc"] for r in products_df.select(["id_producto","desc"]).iter_rows(named=True)}
fam_map = {r["id_producto"]: r["familia1"] for r in products_df.select(["id_producto","familia1"]).iter_rows(named=True)}

In [4]:
# --- Vocabularios y arquitectura (idéntico a 06/07) ---
vocab_ruc = sorted(train_df["RUC"].unique().to_list())
vocab_ciudad = sorted(train_df["CIUDAD"].unique().to_list())
vocab_ruta = sorted(train_df["RUTA"].unique().to_list())
vocab_products = sorted(products_df["id_producto"].unique().to_list())
vocab_marca = sorted(products_df["marca"].unique().to_list())
vocab_familia1 = sorted(products_df["familia1"].unique().to_list())
vocab_familia2 = sorted(products_df["familia2"].unique().to_list())

class QueryTower(tf.keras.Model):
    def __init__(self, vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products, embedding_dim=128, dropout_rate=0.2):
        super().__init__()
        self.ruc_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruc, mask_token=None)
        self.ruc_embedding = tf.keras.layers.Embedding(len(vocab_ruc) + 1, 64, name="ruc_emb")
        self.ciudad_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ciudad, mask_token=None)
        self.ciudad_embedding = tf.keras.layers.Embedding(len(vocab_ciudad) + 1, 16, name="ciudad_emb")
        self.ruta_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_ruta, mask_token=None)
        self.ruta_embedding = tf.keras.layers.Embedding(len(vocab_ruta) + 1, 32, name="ruta_emb")
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="product_emb")
        self.geo_normalization = tf.keras.layers.Normalization(axis=-1)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="query_projection")])
    def call(self, inputs):
        ruc_emb = self.ruc_embedding(self.ruc_lookup(inputs["RUC"]))
        ciudad_emb = self.ciudad_embedding(self.ciudad_lookup(inputs["CIUDAD"]))
        ruta_emb = self.ruta_embedding(self.ruta_lookup(inputs["RUTA"]))
        product_emb = self.product_embedding(self.product_lookup(inputs["COD_PROD"]))
        lat = tf.expand_dims(inputs["LATITUD"], axis=-1); lon = tf.expand_dims(inputs["LONGITUD"], axis=-1)
        geo_norm = self.geo_normalization(tf.concat([lat, lon], axis=-1))
        return tf.math.l2_normalize(self.mlp(tf.concat([ruc_emb, ciudad_emb, ruta_emb, product_emb, geo_norm], axis=-1)), axis=-1)

class CandidateTower(tf.keras.Model):
    def __init__(self, vocab_products, vocab_marca, vocab_familia1, vocab_familia2, embedding_dim=128, dropout_rate=0.2):
        super().__init__()
        self.product_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_products, mask_token=None)
        self.product_embedding = tf.keras.layers.Embedding(len(vocab_products) + 1, 64, name="candidate_product_emb")
        self.marca_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_marca, mask_token=None)
        self.marca_embedding = tf.keras.layers.Embedding(len(vocab_marca) + 1, 16, name="candidate_marca_emb")
        self.fam1_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia1, mask_token=None)
        self.fam1_embedding = tf.keras.layers.Embedding(len(vocab_familia1) + 1, 16, name="candidate_fam1_emb")
        self.fam2_lookup = tf.keras.layers.StringLookup(vocabulary=vocab_familia2, mask_token=None)
        self.fam2_embedding = tf.keras.layers.Embedding(len(vocab_familia2) + 1, 16, name="candidate_fam2_emb")
        self.continuous_normalization = tf.keras.layers.Normalization(axis=-1)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(128, activation="relu"), tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(embedding_dim, name="candidate_projection")])
    def call(self, inputs):
        prod_emb = self.product_embedding(self.product_lookup(inputs["id_producto"]))
        marca_emb = self.marca_embedding(self.marca_lookup(inputs["marca"]))
        fam1_emb = self.fam1_embedding(self.fam1_lookup(inputs["familia1"]))
        fam2_emb = self.fam2_embedding(self.fam2_lookup(inputs["familia2"]))
        precio = tf.expand_dims(inputs["precio"], axis=-1); peso = tf.expand_dims(inputs["peso_unitario"], axis=-1)
        cont_norm = self.continuous_normalization(tf.concat([precio, peso], axis=-1))
        return tf.math.l2_normalize(self.mlp(tf.concat([prod_emb, marca_emb, fam1_emb, fam2_emb, cont_norm], axis=-1)), axis=-1)

In [5]:
def adapt_and_init(qt, ct):
    geo_train = train_df.select(["LATITUD", "LONGITUD"]).sample(n=100_000, seed=42).to_numpy().astype(np.float32)
    qt.geo_normalization.adapt(geo_train)
    cont_train = products_df.select(["precio", "peso_unitario"]).to_numpy().astype(np.float32)
    ct.continuous_normalization.adapt(cont_train)
    dq = {"RUC": tf.constant([vocab_ruc[0]]), "CIUDAD": tf.constant([vocab_ciudad[0]]),
          "RUTA": tf.constant([vocab_ruta[0]]), "LATITUD": tf.constant([0.0], dtype=tf.float32),
          "LONGITUD": tf.constant([0.0], dtype=tf.float32), "COD_PROD": tf.constant([vocab_products[0]])}
    dc = {"id_producto": tf.constant([vocab_products[0]]), "marca": tf.constant([vocab_marca[0]]),
          "familia1": tf.constant([vocab_familia1[0]]), "familia2": tf.constant([vocab_familia2[0]]),
          "precio": tf.constant([0.0], dtype=tf.float32), "peso_unitario": tf.constant([0.0], dtype=tf.float32)}
    _ = qt(dq); _ = ct(dc)

def candidate_embeddings(ct, pdf):
    ds = tf.data.Dataset.from_tensor_slices({
        "id_producto": pdf["id_producto"].to_numpy(), "marca": pdf["marca"].to_numpy(),
        "familia1": pdf["familia1"].to_numpy(), "familia2": pdf["familia2"].to_numpy(),
        "precio": pdf["precio"].to_numpy().astype(np.float32),
        "peso_unitario": pdf["peso_unitario"].to_numpy().astype(np.float32)}).batch(1024)
    return tf.concat([ct(b) for b in ds], axis=0)

def query_embeddings(qt, qdf):
    ds = tf.data.Dataset.from_tensor_slices({
        "RUC": qdf["RUC"].to_numpy(), "CIUDAD": qdf["CIUDAD"].to_numpy(), "RUTA": qdf["RUTA"].to_numpy(),
        "LATITUD": qdf["LATITUD"].to_numpy().astype(np.float32),
        "LONGITUD": qdf["LONGITUD"].to_numpy().astype(np.float32),
        "COD_PROD": qdf["COD_PROD"].to_numpy()}).batch(1024)
    return tf.concat([qt(b) for b in ds], axis=0)

In [6]:
query_tower = QueryTower(vocab_ruc, vocab_ciudad, vocab_ruta, vocab_products)
candidate_tower = CandidateTower(vocab_products, vocab_marca, vocab_familia1, vocab_familia2)
adapt_and_init(query_tower, candidate_tower)
query_tower.load_weights("models/query_tower.weights.h5")
candidate_tower.load_weights("models/candidate_tower.weights.h5")
cand_emb = candidate_embeddings(candidate_tower, products_df).numpy()
print("Embeddings de catalogo:", cand_emb.shape)

Embeddings de catalogo: (20683, 128)


## 1. Contexto representativo (documentado)

In [7]:
ctx_ciudad = train_df["CIUDAD"].mode().to_list()[0]
ctx_ruta = train_df["RUTA"].mode().to_list()[0]
ctx_lat = float(train_df["LATITUD"].median()); ctx_lon = float(train_df["LONGITUD"].median())
ctx_ruc = train_df["RUC"].value_counts(sort=True)["RUC"].to_list()[0]
print(f"Contexto fijo -> RUC={ctx_ruc}, CIUDAD={ctx_ciudad}, RUTA={ctx_ruta}, "
      f"LAT={ctx_lat:.2f}, LON={ctx_lon:.2f}")

Contexto fijo -> RUC=CLIENTE_11714, CIUDAD=uio, RUTA=3 SUR 2, LAT=-0.33, LON=-78.55


## 2. Selección de productos-semilla (búsqueda en descripción)

In [8]:
import re
seed_patterns = {
    "Cemento": r"CEMENTO",
    "Tubería PVC desagüe": r"(TUBO|TUBERIA).*PVC|PVC.*DESAG",
    "Varilla corrugada": r"VARILLA",
    "Cable eléctrico": r"CABLE.*(THHN|FLEX|#?\s*1[02])|CABLE",
    "Pintura látex": r"(PINTURA|LATEX|CAUCHO)",
}
seeds = {}
desc_df = products_df.select(["id_producto", "desc", "familia1"])
for label, pat in seed_patterns.items():
    hit = desc_df.filter(pl.col("desc").str.contains(f"(?i){pat}"))
    if hit.height:
        row = hit.row(0, named=True)
        seeds[label] = row["id_producto"]
        print(f"{label:24} -> {row['id_producto']:14} | {row['desc']}")
    else:
        print(f"{label:24} -> SIN COINCIDENCIA")

Cemento                  -> PE-CE329       | CEMENTO BLANCO TOLT FUNDA 5KG
Tubería PVC desagüe      -> 11250006608    | TUBO PVC SCH-CEDU 40 2"X6MTR
Varilla corrugada        -> VA-HI028       | ADELCA VARILLA CORRUG. 28MMX12
Cable eléctrico          -> 1038087        | CABLE USB 3 EN 1  CA-830
Pintura látex            -> PI-PI10853     | PINT.LATEX  PLASC.BABY ROSS GL


## 3. Top-K real por semilla

In [9]:
def recommend(seed_id, k=10):
    q = {"RUC": tf.constant([ctx_ruc]), "CIUDAD": tf.constant([ctx_ciudad]),
         "RUTA": tf.constant([ctx_ruta]), "LATITUD": tf.constant([ctx_lat], dtype=tf.float32),
         "LONGITUD": tf.constant([ctx_lon], dtype=tf.float32), "COD_PROD": tf.constant([seed_id])}
    qv = query_tower(q).numpy()[0]
    scores = cand_emb @ qv
    order = np.argsort(-scores)
    recs = [catalog_arr[j] for j in order if catalog_arr[j] != seed_id][:k]
    return recs

records = []
for label, sid in seeds.items():
    recs = recommend(sid, k=10)
    print(f"\n=== {label}  (semilla: {desc_map.get(sid,'?')}) ===")
    for rid in recs:
        print(f"   - [{fam_map.get(rid,'?'):16}] {desc_map.get(rid,'?')}")
    records.append({"semilla_label": label, "semilla_id": sid, "semilla_desc": desc_map.get(sid,"?"),
                    "top_k_desc": " | ".join(desc_map.get(r,"?") for r in recs[:6])})
table18_real = pd.DataFrame(records)
table18_real.to_csv("experimentos/table18_real_topk.csv", index=False)
print("\nGuardado experimentos/table18_real_topk.csv")


=== Cemento  (semilla: CEMENTO BLANCO TOLT FUNDA 5KG) ===
   - [FERRETERIA      ] VIRUTA #6 DON BRILLO 10 UND
   - [FERRETERIA      ] DECORLAC SELLADOR CAT.GL
   - [FERRETERIA      ] VIRUTA #5 DON BRILLO 10 UND
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE LTS.
   - [FERRETERIA      ] DECORLAC SELLADOR CATALI.LITRO
   - [FERRETERIA      ] PINT.CAUCHO PLASC.BLANCO GALON
   - [FERRETERIA      ] CILINDRO GLP 15KG-AGIPGAS
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE GLN.
   - [FERRETERIA      ] BROCA ITALY/ALMAN CONCRET 3/8
   - [FERRETERIA      ] DECORLAC FOND.CAT.BLANCO LITRO



=== Tubería PVC desagüe  (semilla: TUBO PVC SCH-CEDU 40 2"X6MTR) ===
   - [FERRETERIA      ] VIRUTA #6 DON BRILLO 10 UND
   - [FERRETERIA      ] VIRUTA #5 DON BRILLO 10 UND
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE LTS.
   - [FERRETERIA      ] DECORLAC SELLADOR CAT.GL
   - [FERRETERIA      ] DECORLAC SELLADOR CATALI.LITRO
   - [FERRETERIA      ] PINT.CAUCHO PLASC.BLANCO GALON
   - [FERRETERIA      ] CILINDRO GLP 15KG-AGIPGAS
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE GLN.
   - [FERRETERIA      ] BROCA ITALY/ALMAN CONCRET 3/8
   - [FERRETERIA      ] TORNILLO MDF 8X1-1/2



=== Varilla corrugada  (semilla: ADELCA VARILLA CORRUG. 28MMX12) ===
   - [FERRETERIA      ] VIRUTA #6 DON BRILLO 10 UND
   - [FERRETERIA      ] DECORLAC SELLADOR CAT.GL
   - [FERRETERIA      ] VIRUTA #5 DON BRILLO 10 UND
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE LTS.
   - [FERRETERIA      ] DECORLAC SELLADOR CATALI.LITRO
   - [FERRETERIA      ] PINT.CAUCHO PLASC.BLANCO GALON
   - [FERRETERIA      ] CILINDRO GLP 15KG-AGIPGAS
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE GLN.
   - [FERRETERIA      ] BROCA ITALY/ALMAN CONCRET 3/8
   - [FERRETERIA      ] DECORLAC FOND.CAT.BLANCO LITRO



=== Cable eléctrico  (semilla: CABLE USB 3 EN 1  CA-830) ===
   - [FERRETERIA      ] VIRUTA #6 DON BRILLO 10 UND
   - [FERRETERIA      ] DECORLAC SELLADOR CAT.GL
   - [FERRETERIA      ] VIRUTA #5 DON BRILLO 10 UND
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE LTS.
   - [FERRETERIA      ] DECORLAC SELLADOR CATALI.LITRO
   - [FERRETERIA      ] PINT.CAUCHO PLASC.BLANCO GALON
   - [FERRETERIA      ] CILINDRO GLP 15KG-AGIPGAS
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE GLN.
   - [FERRETERIA      ] BROCA ITALY/ALMAN CONCRET 3/8
   - [FERRETERIA      ] DECORLAC FOND.CAT.BLANCO LITRO

=== Pintura látex  (semilla: PINT.LATEX  PLASC.BABY ROSS GL) ===
   - [FERRETERIA      ] DECORLAC SELLADOR CAT.GL
   - [FERRETERIA      ] VIRUTA #6 DON BRILLO 10 UND
   - [FERRETERIA      ] DECORLAC SELLADOR CATALI.LITRO
   - [FERRETERIA      ] ANTICORR.DURAC.NEGRO MATE LTS.
   - [FERRETERIA      ] VIRUTA #5 DON BRILLO 10 UND
   - [FERRETERIA      ] DECORLAC FOND.CAT.BLANCO LITRO
   - [FERRETERIA      


Guardado experimentos/table18_real_topk.csv


## Conclusión

La tabla `table18_real_topk.csv` contiene recuperaciones reales del modelo y debe sustituir (o acompañar, etiquetada como salida real) a la Tabla 18 ilustrativa del Capítulo 3. El contexto fijo está documentado, cumpliendo la trazabilidad exigida por la integridad de investigación.